# 256^3 forced RMHD turbulence on a Kaggle P100

Single-GPU, single-process run (no MPI decomposition -- see the discussion in the project
thread this notebook came from: a single P100 only has one device, so real multi-rank
parallelism doesn't apply here). Runs forced RMHD turbulence at 256^3 to t=20, saves full-state
checkpoints periodically, then computes and saves perpendicular + parallel energy spectra for
every saved snapshot, and zips everything into one archive in `/kaggle/working` for download.

**Before running:** in the notebook Settings panel, select **GPU: P100** as the accelerator and
turn **Internet on** (needed to `pip install`/`apt-get install` the packages below).

**Kaggle constraints this notebook is designed around:**
- `/kaggle/working` (the persisted, downloadable output) is capped at 20GiB *and* ~500 files.
  Orbax checkpoints are made of many small files per snapshot, so this notebook does the actual
  run and checkpointing in `/kaggle/tmp` (large, unlimited-file scratch space that isn't
  persisted) and only writes a single zip file into `/kaggle/working` at the end.
- Free P100 sessions are time- and quota-limited (session length + ~30 GPU-hours/week as of
  when this was written -- check Kaggle's current limits, this changes over time). Whether
  t=20 at 256^3 finishes in one session depends on the actual per-step wall-clock cost on a
  P100, which we have not benchmarked -- watch the printed `t` progress after the first few
  blocks below to extrapolate an ETA before committing to the full run.

## 1. System setup

`mpi4py` needs an actual MPI implementation to build against even though we never use more than
one rank -- it's a hard import dependency of this package (see `jax_rmhd/config.py`).

In [ ]:
!apt-get update -qq && apt-get install -y -qq libopenmpi-dev openmpi-bin
!pip install -q mpi4py mpi4jax orbax-checkpoint tensorstore
# Match this to Kaggle's current CUDA version if this doesn't pick up a GPU-enabled jaxlib --
# check with `!nvidia-smi` and https://docs.jax.dev/en/latest/installation.html if needed.
!pip install -q -U "jax[cuda12]"

## 2. Get the jax_rmhd package onto this instance

Pick ONE of the two cells below depending on how you've made the repo available to this kernel.
Default assumes you've attached it as a Kaggle Dataset (Add Data -> Upload -> point at your
`jax_rmhd` checkout) -- that's the most reliable way to bring custom code to Kaggle without
depending on external network/repo-visibility details this notebook can't know in advance.

In [ ]:
# Option A (default): repo attached as a Kaggle Dataset.
# EDIT ME: match this to whatever your dataset is actually named/mounted as.
import sys, os
REPO_PATH = "/kaggle/input/jax-rmhd/jax_rmhd"
assert os.path.isdir(REPO_PATH), f"{REPO_PATH} not found -- check your attached dataset's path"
!pip install -q -e {REPO_PATH}

In [ ]:
# Option B: clone directly from a git URL instead (uncomment and edit if you're not using a
# Dataset). Requires the repo to actually be reachable from Kaggle's network.
# !git clone <your-repo-url> /kaggle/working/jax_rmhd_src
# !pip install -q -e /kaggle/working/jax_rmhd_src

In [ ]:
import os
os.environ["RMHD_PRECISION"] = "64"  # matches convention used throughout this project's own
                                       # tests/examples; P100 has decent FP64 throughput unlike
                                       # cheaper cards, so there's little reason to drop to 32-bit
import jax
import jax_rmhd as jr
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import jax_rmhd.snapshot_io as sn
from jax_rmhd import diagnostics as diag
from jax_rmhd.physics import shared_physics

jr.init_cluster()
print("JAX devices:", jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU visible -- check the accelerator setting".

## 3. Parameters

Forcing/dissipation values match the 64^3 reference run this notebook was scaled up from
(`tests/forced_turbulence_64cubed.py`), just at 4x the linear resolution. **The dissipation
coefficients (`visc`, `res`) have NOT been re-tuned for 256^3** -- at higher resolution the same
coefficients may under- or over-dissipate relative to where you'd want the dissipation range to
sit. Watch the `<psi^2>` and energy time series computed in the analysis section below: per the
zeroth law of turbulence, `visc`/`res` set the time to reach saturation and the dissipation-range
cutoff, not the saturated amplitude itself -- if `<psi^2>` is still climbing at t=20 rather than
plateaued, the run hasn't saturated yet and t_end and/or dissipation should be revisited.

`t_snap=1.0` (~22 full-state snapshots over t=20) is deliberately coarser than the 0.25 used in
the smaller reference runs -- at 256^3/complex128 each snapshot is ~258MiB, and finer spacing
(e.g. 0.25 -> ~82 snapshots) would land close to Kaggle's 20GiB output cap. Spectra are cheap
regardless of snapshot count, so widen this if you want closer temporal coverage of the field
dynamics and are willing to trim it back before the final zip, or narrow the field-state cadence
independently from a finer spectra cadence if you extend this notebook.

In [ ]:
nx = ny = nz = 256
Lx = Ly = Lz = 2.0 * jnp.pi

t_snap = 1.0
t_end = 20.0
cfl_safety = 0.5
spatial_dimensions = 3

visc = 1e-5
res = 1e-5
hyper = 3

forcing = True
forcing_mode = "elsasser"
forcing_power_elsasser = (0.3, 0.3)
forcing_tau = 1.0
fshell = (1, 3)
forcing_seed = 42
forcing_scale_max = 1.0

params = jr.Parameters(nx=nx, ny=ny, Lx=Lx, Ly=Ly, nz=nz, Lz=Lz, diss=(visc, res),
                        hyper=hyper, cfl_safety=cfl_safety, dims=spatial_dimensions,
                        forcing=forcing, forcing_mode=forcing_mode,
                        forcing_power_elsasser=forcing_power_elsasser,
                        forcing_tau=forcing_tau, fshell=fshell, forcing_seed=forcing_seed,
                        forcing_scale_max=forcing_scale_max)
kgrid = jr.setup_kgrids(params)
print(f"params.size={params.size} (single-GPU run -- z-domain is not decomposed)")

## 4. Run

Checkpoints and all raw run output go to `/kaggle/tmp` (large scratch, not persisted) rather
than `/kaggle/working`, to sidestep the output file-count cap -- see the notes at the top.

In [ ]:
scratch_root = "/kaggle/tmp/forced_turbulence_256cubed"
snap_path = os.path.join(scratch_root, "checkpoints")
spectra_dir = os.path.join(scratch_root, "spectra")
os.makedirs(spectra_dir, exist_ok=True)

mngr = jr.snapshot_manager_setup(params, snap_path=snap_path, nsnap=30)

def zero_init(x, y, z):
    return jnp.zeros((2,) + jnp.broadcast_shapes(x.shape, y.shape, z.shape))

state = jr.initialize(zero_init, params)

nblock = jr.estimate_good_nblock(state, kgrid, params, t_snap, t_end, nblock_min=10)
print("nblock estimate:", nblock)

In [ ]:
# Prints state.t after every block -- watch the first handful of prints to gauge wall-clock
# per unit of simulated time and extrapolate whether t_end=20 will fit in this session.
end_state = jr.simulate_scan(state, kgrid, params, nblock, t_snap=t_snap, t_end=t_end,
                              mngr=mngr, save=True)

## 5. Spectra for every saved snapshot

Reloads each checkpoint and computes both the perpendicular spectrum (`perpspec`) and the
parallel/z spectrum (`parspec` -- only valid for a single-rank z-domain, which this run is by
construction) for it, plus a lightweight combined time series of kinetic/magnetic energy and
mean-square flux function `<psi^2>` across all snapshots (the saturation diagnostic mentioned
above). Uses `sn.get_saved_steps`/`sn.load_snapshot`'s plain-path API, not a CheckpointManager
directly -- see the project thread this notebook came from for why.

In [ ]:
steps = sorted(sn.get_saved_steps(snap_path))
print(f"{len(steps)} snapshots saved:", steps)

summary_t, summary_Ekin, summary_Emag, summary_psisq = [], [], [], []

for isnap in steps:
    snap = sn.load_snapshot(isnap, snap_path, params)
    bins_perp, spec_u_perp, spec_b_perp = diag.perpspec(snap, kgrid, params)
    bins_par, spec_u_par, spec_b_par = diag.parspec(snap, kgrid, params)

    phik, psik = snap.fields[0], snap.fields[1]
    E_kin = 0.5 * float(shared_physics.perp_inner_product(phik, phik, kgrid, params))
    E_mag = 0.5 * float(shared_physics.perp_inner_product(psik, psik, kgrid, params))
    psi_sq = float(shared_physics.perp_mean_square(psik, psik, kgrid, params))

    summary_t.append(float(snap.t))
    summary_Ekin.append(E_kin)
    summary_Emag.append(E_mag)
    summary_psisq.append(psi_sq)

    np.savez(os.path.join(spectra_dir, f"spectra_{isnap:04d}.npz"),
             t=float(snap.t),
             k_perp=np.asarray(bins_perp), E_u_perp=np.asarray(spec_u_perp), E_b_perp=np.asarray(spec_b_perp),
             k_par=np.asarray(bins_par), E_u_par=np.asarray(spec_u_par), E_b_par=np.asarray(spec_b_par))
    print(f"  isnap={isnap:3d}  t={float(snap.t):7.3f}  E_kin={E_kin:.4e}  E_mag={E_mag:.4e}  psi_sq={psi_sq:.4e}")

np.savez(os.path.join(spectra_dir, "summary_timeseries.npz"),
         t=np.array(summary_t), E_kin=np.array(summary_Ekin),
         E_mag=np.array(summary_Emag), psi_sq=np.array(summary_psisq))
print(f"wrote {len(steps)} per-snapshot spectra files + 1 summary file to {spectra_dir}")

## 6. Quick look

Sanity-check plots before downloading -- the actual per-snapshot spectra files have everything
needed to reproduce these (and more) after download.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(summary_t, summary_Ekin, marker='o', markersize=3, label=r'$E_{kin}$')
axes[0].plot(summary_t, summary_Emag, marker='o', markersize=3, label=r'$E_{mag}$')
axes[0].plot(summary_t, summary_psisq, marker='o', markersize=3, label=r'$\langle\psi^2\rangle$')
axes[0].set_xlabel('t'); axes[0].legend(); axes[0].set_title('Energy / saturation diagnostics vs t')

last = np.load(os.path.join(spectra_dir, f"spectra_{steps[-1]:04d}.npz"))
axes[1].loglog(last["k_perp"], last["E_u_perp"], label=r'$E_u(k_\perp)$')
axes[1].loglog(last["k_perp"], last["E_b_perp"], label=r'$E_b(k_\perp)$')
kunit = min(2*jnp.pi/params.Lx, 2*jnp.pi/params.Ly)
axes[1].axvline(params.fshell[0]*kunit, color='k', linestyle=':', label='forcing shell')
axes[1].axvline(params.fshell[1]*kunit, color='k', linestyle=':')
axes[1].set_xlabel(r'$k_\perp$'); axes[1].legend()
axes[1].set_title(f'Perpendicular spectrum at t={float(last["t"]):.2f} (final snapshot)')

plt.tight_layout()
plt.show()

## 7. Zip everything for download

Single zip file into `/kaggle/working` -- counts as one item against the output file cap
regardless of how many individual checkpoint/spectra files are inside it.

In [ ]:
import zipfile

zip_path = "/kaggle/working/forced_turbulence_256cubed_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(scratch_root):
        for fn in files:
            full = os.path.join(root, fn)
            arc = os.path.relpath(full, scratch_root)
            zf.write(full, arc)

with zipfile.ZipFile(zip_path) as zf:
    bad = zf.testzip()
    assert bad is None, f"corrupt entry in zip: {bad}"
    n_entries = len(zf.namelist())

zip_size_gib = os.path.getsize(zip_path) / 1024**3
print(f"Wrote and verified {zip_path}")
print(f"  {n_entries} entries, {zip_size_gib:.3f} GiB (Kaggle /kaggle/working cap is 20 GiB)")